In [7]:
# Import necessary packages

import torch
import torch.nn as nn  # Contains all functions & modules for neural networks
import torch.optim as optim  # For optimizing neural network
from torch.utils.data import DataLoader, random_split  # Helps to take Kaggle data and load it into network

from torchvision import datasets, transforms  # For manipulations in the dataset
from PIL import Image  # For manipulations/transformations to the image

import cv2
import time

# Kaggle Dataset: https://www.kaggle.com/datasets/fahadullaha/facial-emotion-recognition-dataset

In [8]:
# Configurations

DATA_DIRECTORY = "./processed_data"
IMAGE_SIZE = 64
EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 0.002
VALIDATION_SPLIT = 0.2
MODEL_PATH = "./model.pth"

In [9]:
device = torch.device("cuda")  # Creates a device object representing an NVIDIA GPU because we are using the GPU
#print(device)

In [10]:
# Building the CNN model

class CNN(nn.Module):
  def __init__(self, num_classes):
    super().__init__()  # Required to run so PyTorch can properly register layers and parameters

    # Extract features from images (nn.Sequential is a container that applies layers in order)
    self.features = nn.Sequential(
        # Convolution layer: 3 input channels (RGB), 16 output channels (16 feature maps)
        nn.Conv2d(3, 16, kernel_size=3, padding=1),
        # ReLU activation fn: replaces negatives w/ 0
        # Downsample by a factor of 2 (e.g., (16, 32, 32) --> (16, 16, 16)) & keep strongest features
        nn.ReLU(), nn.MaxPool2d(2),
        # Keep repeating process for other layers; output of previous layer is input to next layer
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(), nn.MaxPool2d(2)
    )

    # Turns extracted features into predictions
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(4096, 128),
        nn.ReLU(),
        nn.Dropout(0.4),  # Randomly zero-out 40% of neurons during training (helps prevent overfitting)
        nn.Linear(128, num_classes)
    )

  # How data flows forward through the network
  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [11]:
# Data preprocessing + augmentation
# Prepares images before going into the CNN

# Data for training
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),  # Randomly flip image to help the model generalize better
    transforms.ToTensor(),  # Converts to PyTorch tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalizes each channel separately; values means and stds used by many from ImageNet statistics
])

# Data during validation/testing
validation_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [13]:
# Extract files from zip file containing our dataset
import zipfile
import os
zip_path = 'emotion-detection-dataset.zip'
extraction_path = './'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall(extraction_path)

In [14]:
# Creates dataset from a structured folder of images
full_dataset = datasets.ImageFolder(DATA_DIRECTORY, transform=train_transform)
print("Number of images in dataset: ", len(full_dataset))

Number of images in dataset:  49779


In [15]:
# Check the classes we have in the dataset
class_names = full_dataset.classes
num_classes = len(class_names)

print("Class names: ", class_names)
print("Number of classes: ", num_classes)

Class names:  ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Number of classes:  7


In [16]:
# Split the data into its train/test split
validation_count = int(len(full_dataset) * VALIDATION_SPLIT)
train_count = int(len(full_dataset) - validation_count)

train, val = random_split(full_dataset, [train_count, validation_count])

print("Num of training samples: ", train_count)
print("Num of validation samples: ", validation_count)

Num of training samples:  39824
Num of validation samples:  9955


In [17]:
# Perform data preprocessing + augmentation on validation data
val.dataset.transform = validation_transform

In [18]:
# Feed your data into the model

# DataLoader() turns dataset into batches
# num_workers --> Number fo CPU processes used to laod data in parallel
# pin_memory --> Speeds up transfer from CPU to GPU
train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True, num_workers=12, pin_memory=True)
val_loader = DataLoader(val, batch_size=BATCH_SIZE, shuffle=True, num_workers=12, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [19]:
# Build instance of the model
model = CNN(num_classes=num_classes).to(device)  # .to(device) shifts it to using GPU
criterion = nn.CrossEntropyLoss()  # Calculating the loss (using cross entropy)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)  # Updates model weights to reduce loss

In [20]:
# Building the training loop!

def train():
  best_accuracy = 0.0

  for epoch in range(1, EPOCHS + 1):
    model.train()  # Puts model in training mode
    avg_loss = 0.0  # Used to accumulate total training loss
    avg_correct = 0  # Used to count number of correct predictions
    total = 0  # Used to count total number of samples seen
    t0 = time.time()  # Start time for the epoch

    for images, labels in train_loader:
      # Moves data to GPU
      images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

      optimizer.zero_grad()  # Clear gradients from previous iteration; PyTorch accumulates them by default
      outputs = model(images)  # Forward pass --> pass images through CNN to get predictions
      loss = criterion(outputs, labels)  # Measure how wrong preds are
      loss.backward()  # Backward pass --> Computes gradients using backpropagation
      optimizer.step()  #Updates model parameters using gradients

      avg_loss += loss.item() * images.size(0)  # Helps to compute avg loss later: loss.item() is scalar loss for the batch & we multiply by batch size to get total loss contribution
      preds = outputs.argmax(dim=1)  # Picks the class with the highest score for pred
      avg_correct += (preds == labels).sum().item()  # Helps to compute avg correct later; counts how many preds were corect
      total += images.size(0)  # Tracks total num of samples processed

    model.eval()  # Puts model in evaluation mode
    validation_loss = 0.0
    validation_correct = 0
    validation_total = 0

    with torch.no_grad():  # Prevents gradient calculation (saves memory, speeds up validation, & b/c no training is happening here)
      for images, labels in val_loader:
        # Moves data to GPU
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        # Forward pass + loss (no back prop here)
        outputs = model(images)
        loss = criterion(outputs, labels)

        validation_loss += loss.item() * images.size(0)  # Helps to compute val loss later; accumulates total val loss
        preds = outputs.argmax(dim=1)
        validation_correct += (preds == labels).sum().item()  # Counts correct preds
        validation_total += images.size(0)  # Counts total samples

    # Training metrics (using partial calculations from earlier)
    avg_loss /= total
    avg_accuracy = avg_correct / total

    # Validation metrics (using partial calculations from earlier)
    validation_loss /= validation_total
    validation_accuracy = validation_correct / validation_total

    t1 = time.time()
    elapsed_time = t1 - t0  # Measures how long the epoch took

    print(f"EPOCH {epoch} / {EPOCHS}  train_loss={avg_loss:.3f}   train_accuracy={avg_accuracy:.3f}   validation_loss={validation_loss:.3f}   validation_accuracy={validation_accuracy:.3f}   time={elapsed_time:.1f}sec")

    # Only if the model improves during each epoch, update the best score & save the model
    if validation_accuracy > best_accuracy:
      best_accuracy = validation_accuracy
      torch.save({
          "model_state" : model.state_dict(),
          "class_names": class_names,
          "img_size": IMAGE_SIZE
      }, MODEL_PATH)

      print(f"Saving model with best validation accuracy: {best_accuracy:.3f} to {MODEL_PATH}")

  print("Training completed!")


In [21]:
# Build the 'Emotion Detector' video
# Takes the live video --> preprocesses each frame --> runs it through the CNN --> shows prediction on screen

def inference():
  checkpoint = torch.load(MODEL_PATH, map_location=device)  # Load the saved checkpoint from disk
  model.load_state_dict(checkpoint["model_state"])  # Load learned weights into model (restores what the model learned during training)
  model.to(device)  # Moves model to GPU
  model.eval()  # Set model to evaluation mode
  class_names = checkpoint["class_names"]  # Loads label names; used to convert prediction index to a readable label

  print("Webcam is starting...press q to quit")
  webcam = cv2.VideoCapture(0)  # 0 is default camera
  if not webcam.isOpened():
    print("Webcam Error")
    return

  while True:  # Process video continuously
    ret, frame = webcam.read()  # Captures a frame from the webcam

    if not ret:
      break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # OpenCV uses BGR, but PyTorch expects RGB
    resized_image = Image.fromarray(rgb).resize((IMAGE_SIZE, IMAGE_SIZE))  # Resize image to match training size iamges
    tensor = transforms.functional.to_tensor(resized_image)  # Converts image to tensor
    tensor = transforms.functional.normalize(tensor, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Apply same normalization as training so that it's consistent and is able to be classified correctly
    tensor = tensor.unsqueeze(0).to(device)  # Adds batch dimension and moves tensor to GPU

    with torch.no_grad():  # Disable gradient calculation
      output = model(tensor)  # Get outputs from forward pass of CNN
      probs = torch.nn.functional.softmax(output, dim=1)  # Converts outputs to probabilities using softmax activation function
      top_prob, prob_indx = torch.max(probs, dim=1)  # top_prob is the highest probability, prob_indx is the index of the predicted class
      label = class_names[prob_indx.item()]  # Convert the index to a readable label
      confidence = top_prob.item()  # Get the confidence score using the highest probability

    # Output text on the video frame
    cv2.putText(frame, f"{label} {confidence:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    cv2.imshow("Emotion Detector (q to quit)", frame)

    # If the user presses 'q', then quit the video
    if cv2.waitKey(1) & 0xFF == ord("q"):
      break

  webcam.release()
  cv2.destroyAllWindows()

In [22]:
# Run the training loop
train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


EPOCH 1 / 50  train_loss=1.474   train_accuracy=0.431   validation_loss=1.245   validation_accuracy=0.514   time=39.9sec
Saving model with best validation accuracy: 0.514 to ./model.pth
EPOCH 2 / 50  train_loss=1.245   train_accuracy=0.526   validation_loss=1.167   validation_accuracy=0.544   time=37.5sec
Saving model with best validation accuracy: 0.544 to ./model.pth
EPOCH 3 / 50  train_loss=1.187   train_accuracy=0.547   validation_loss=1.132   validation_accuracy=0.563   time=37.9sec
Saving model with best validation accuracy: 0.563 to ./model.pth
EPOCH 4 / 50  train_loss=1.138   train_accuracy=0.562   validation_loss=1.095   validation_accuracy=0.586   time=38.1sec
Saving model with best validation accuracy: 0.586 to ./model.pth
EPOCH 5 / 50  train_loss=1.115   train_accuracy=0.573   validation_loss=1.137   validation_accuracy=0.567   time=38.0sec
EPOCH 6 / 50  train_loss=1.092   train_accuracy=0.580   validation_loss=1.077   validation_accuracy=0.587   time=37.6sec
Saving model w